# 03. Merge Sort

Merge sort is **divide and conquer**: split, sort each half, merge. It is always O(n log n), stable, and the base of many interview problems (Count Inversions, Reverse Pairs, Sort List).

**Learning order:** MERGE STEP → SPLIT → FULL ALGORITHM → TRACE → COMPLEXITY → INVERSIONS → INTERVIEW THINKING

## 1. The key step: merging two sorted lists

Keep one pointer on each list. Take the smaller front element, move that pointer. When one list ends, append the rest of the other.

This is two pointers (same idea as *Merge Sorted Array*). Using `<=` takes the left element first on ties → **stable**.

In [ ]:
def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

print(merge([1, 4, 7], [2, 3, 9]))   # [1, 2, 3, 4, 7, 9]

## 2. The full algorithm

1. If length ≤ 1 → already sorted (base case).
2. Split at `mid`.
3. Sort left, sort right (recursion).
4. Merge them.

The real sorting work happens during the merge, on the way back up.

In [ ]:
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

print(merge_sort([38, 27, 43, 3, 9, 82, 10]))

In [ ]:
# Trace on [38, 27, 43, 3]: indentation = recursion depth
def merge_sort_trace(arr, depth=0):
    pad = "  " * depth
    print(f"{pad}split {arr}")
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort_trace(arr[:mid], depth + 1)
    right = merge_sort_trace(arr[mid:], depth + 1)
    merged = merge(left, right)
    print(f"{pad}merge {left} + {right} -> {merged}")
    return merged

merge_sort_trace([38, 27, 43, 3])

## 3. Complexity

- The array is halved each time → about **log₂ n levels**.
- Each level merges all n elements in total → **O(n) work per level**.
- Total: **O(n log n)** in best, average and worst case.
- Space: **O(n)** for merged lists (+ O(log n) recursion stack).
- Stable: yes. Not in-place.

Slicing `arr[:mid]` copies elements, but that is also O(n) per level, so the total stays O(n log n). (An index-based version avoids copies.)

**Where merge sort shines:** linked lists (merge needs no random access), huge data on disk (external sort), when stability and guaranteed O(n log n) matter.

## 4. Counting inversions with merge sort

Inversion = pair `i < j` with `arr[i] > arr[j]`. Brute force is O(n²).

**Key insight:** while merging, if `right[j] < left[i]`, then `right[j]` is smaller than `left[i]` **and everything after it in `left`** (left is sorted). That adds `len(left) - i` inversions at once.

Inversions inside each half are counted by the recursion. Cross inversions are counted during the merge.

In [ ]:
def count_inversions(arr):
    def sort_count(a):
        if len(a) <= 1:
            return a, 0
        mid = len(a) // 2
        left, lc = sort_count(a[:mid])
        right, rc = sort_count(a[mid:])
        merged, cross = [], 0
        i = j = 0
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1
                cross += len(left) - i     # all remaining left items > right[j]
        merged.extend(left[i:])
        merged.extend(right[j:])
        return merged, lc + rc + cross

    return sort_count(arr)[1]

print(count_inversions([2, 4, 1, 3, 5]))   # 3
print(count_inversions([5, 4, 3, 2, 1]))   # 10

## 5. Interview thinking

**Signals for merge sort:** "count pairs where a[i] > a[j] with i < j", "sort a linked list in O(n log n)", "need stable O(n log n)", "reverse pairs".

**Explain it like this:** "I split, solve each half, and during the merge both halves are sorted, so I can count or combine across them in linear time."

**Common mistakes:** using `<` instead of `<=` (loses stability), forgetting to append leftovers, adding `1` instead of `len(left) - i`, mid calculation off by one.

## Quick Revision

- Split → sort halves → merge. Merge = two pointers on sorted lists.
- O(n log n) always, O(n) extra space, stable (use `<=`).
- Count Inversions: when right element is taken, add `len(left) - i`.
- Use for: linked list sort, inversions, reverse pairs, external sort.